[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Qwen35_0p8B_All24_WindowAnchor_NoCeNN_Colab.ipynb)

# Qwen3.5-0.8B → all-24 causal Window-Anchor Attention (NO CeNN, NO DeltaNet recurrence)

**Research experiment:** 18 pretrained Gated DeltaNet token mixers are replaced by your causal window/anchor attention initialized from the original QKV/convolution/gating weights. All 6 full-GQA mixers are also replaced by causal window/anchor attention initialized from the original Q/K/V/o projections and partial MRoPE. 24 original FFNs and layer norms remain frozen.

**Crucial scientific caveat:** DeltaNet already offers more efficient long-context mixing; window+anchor attention is *not* guaranteed to improve speed or preserve quality. We compare an untouched text-only Qwen3.5-0.8B teacher and the all-24 replacement on identical training, DEV and sealed HOLDOUT text. This notebook is **text-only**: no images or video, no KV-cache decoding. Generation recomputes the full prefix each token. Do not claim a deployable streaming replacement or speed advantage until measured.

### Reproducibility, guardrails and output
- Official source: [Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B) (Apache-2.0). The text model is extracted from the multimodal checkpoint without creating a vision model.
- Exactly **18 DeltaNet + 6 full-attention token mixers replaced**; FFNs and norms frozen and fingerprint checked.
- FP16 inference, LoRA projection training, teacher next-token KL, gold-token NLL, best DEV checkpoint selection.
- Sealed holdout agreement, NLL/perplexity ratio, KL/JS, FPS latency and **manual no-cache chat generation**.
- Standalone self-contained custom model with \`qwen35_window_anchor.py\` loader and strict reload test.
- **ZIP of all statistics automatically downloaded**; model ZIP download optional because it is large.
- Hugging Face upload only if the **explicit quality gate** passes (by default holdout top-1 agreement ≥95%, perplexity ratio ≤1.15, JS ≤0.01, no-cache median speedup ≥0.8). If it fails, export **locally** but DO NOT publish a misleading model. No claim of successful training until the user runs Colab.

Use a GPU runtime (L4/A100 recommended; 16GB T4 may require reducing MAX_TOKENS/GRAD_ACCUM). Training can take considerable time due to 24 block replacements. Everything measured is for short **text** contexts; this does not validate Qwen's native million-token context.


In [ ]:
#@title 1. Setup (Qwen3.5-compatible Transformers)
%pip -q install -U "git+https://github.com/huggingface/transformers.git@main" "accelerate>=1.0" "datasets>=3" "huggingface_hub>=0.34" "safetensors>=0.5"
%pip -q install --no-deps "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@143f99a4bdb527001d0f5f8433f435535656a409"

import os, json, math, random, time, gc, hashlib, copy, shutil, zipfile, traceback
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from datasets import load_dataset
from huggingface_hub import HfApi, login, get_token
from safetensors.torch import save_model
from tinycenn_lm import qwen35_window_anchor as wa

SEED=20261018
MODEL_ID="Qwen/Qwen3.5-0.8B"
WINDOW=128 #@param {type:"integer"}
ANCHORS=16 #@param {type:"integer"}
CHUNK=64 #@param {type:"integer"}
MAX_TOKENS=160 #@param {type:"integer"}
TRAIN_EXAMPLES=100 #@param {type:"integer"}
DEV_EXAMPLES=20 #@param {type:"integer"}
HOLDOUT_EXAMPLES=36 #@param {type:"integer"}
TRAIN_STEPS=120 #@param {type:"integer"}
GRAD_ACCUM=2 #@param {type:"integer"}
LORA_RANK=8 #@param {type:"integer"}
LR_START=0.0008 #@param {type:"number"}
LR_END=0.0001 #@param {type:"number"}
EVAL_EVERY=20
MAX_GRAD_NORM=0.6
DISTILL_TEMP=2.0
GOLD_CE_WEIGHT=0.05
AUTO_DOWNLOAD_STATS=True #@param {type:"boolean"}
AUTO_DOWNLOAD_MODEL_ZIP=False #@param {type:"boolean"}
UPLOAD_IF_QUALIFIED=True #@param {type:"boolean"}
HF_PRIVATE=True #@param {type:"boolean"}
HF_NAME="qwen35-0p8b-all24-window-anchor-nocenn" #@param {type:"string"}
MIN_TEACHER_AGREEMENT=0.95
MAX_PERPLEXITY_RATIO=1.15
MAX_JS=0.01
MIN_SPEEDUP=0.80

BASE=Path("/content/Qwen35_0p8B_All24_WindowAnchor")
ARTIFACTS=BASE/"exports"
REPORTS=BASE/"results"
ARTIFACTS.mkdir(parents=True,exist_ok=True)
REPORTS.mkdir(parents=True,exist_ok=True)
assert WINDOW>=16 and 0<=ANCHORS<=WINDOW and CHUNK>=16 and MAX_TOKENS>WINDOW
assert 0<LR_END<LR_START and TRAIN_STEPS>0 and GRAD_ACCUM>0
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),"Enable GPU in Runtime > Change runtime type"
device=torch.device("cuda",torch.cuda.current_device())
torch.cuda.manual_seed_all(SEED)
print("GPU:",torch.cuda.get_device_name(),"source",MODEL_ID)
print("Experiment: replace 18 DeltaNet AND 6 full attention. No CeNN.")
# Cheap CPU causality smoke test before downloading the 1.75GB teacher weights.
a=torch.randn(1,2,10,8)
b=torch.randn(1,2,10,8)
v=torch.randn(1,2,10,8)
ref=wa.causal_window_anchor(a,b,v,None,window=3,anchors=2,chunk_size=4)
v[:,:,8:]+=1000
out=wa.causal_window_anchor(a,b,v,None,window=3,anchors=2,chunk_size=4)
torch.testing.assert_close(ref[:,:,:8],out[:,:,:8],atol=1e-5,rtol=1e-5)
print("PASS: causal window/anchor mixer has no future-token leakage.")



In [ ]:
#@title 2. Separate train / development / sealed holdout text without overlap
# Wikitext supplies real, non-synthetic next-token prediction data.
# If dataset loading fails, use a deterministic, disclosed synthetic fallback.
DATA_SOURCE="wikitext-2-raw-v1"
try:
    ds=load_dataset("Salesforce/wikitext","wikitext-2-raw-v1",split="train")
    unique=[]
    seen=set()
    for row in ds:
        s=" ".join(str(row["text"]).split())
        if 80<=len(s)<=850 and not s.startswith("=") and s not in seen:
            seen.add(s)
            unique.append(s)
    assert len(unique)>=TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES
except Exception as exc:
    DATA_SOURCE="synthetic_fallback"
    print("Real dataset load unavailable:",repr(exc))
    subjects=["A spacecraft","The library","A small river","The scientist",
              "The teacher","The garden","A new algorithm","The weather",
              "An electric train","The city","A musician","An old telescope"]
    verbs=["revealed","described","examined","reconsidered","improved","discovered",
           "measured","analyzed","discussed","investigated","demonstrated"]
    endings=[
      "a surprising result that changed how people understood the problem.",
      "the effects of several different design choices across many experiments.",
      "a practical solution that was tested under controlled conditions.",
      "the surprising history of decisions made during a difficult project.",
      "an important difference between prediction accuracy and computational speed."
    ]
    unique=[f"{a} {v} {c} The summary compared earlier evidence with the final results."
            for a in subjects for v in verbs for c in endings]
rng=random.Random(SEED)
rng.shuffle(unique)
train_text=unique[:TRAIN_EXAMPLES]
dev_text=unique[TRAIN_EXAMPLES:TRAIN_EXAMPLES+DEV_EXAMPLES]
holdout_text=unique[TRAIN_EXAMPLES+DEV_EXAMPLES:TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES]
assert len(holdout_text)==HOLDOUT_EXAMPLES
assert not (set(train_text)&set(dev_text) or set(train_text)&set(holdout_text)
            or set(dev_text)&set(holdout_text))
print("Data source:",DATA_SOURCE,"train",len(train_text),"dev",len(dev_text),"sealed holdout",len(holdout_text))


In [ ]:
#@title 3. Official Qwen3.5 text-only teacher; cache train and DEV logits once
teacher,tokenizer,source_root=wa.load_official_text(MODEL_ID,dtype=torch.float16)
teacher.to(device).eval().requires_grad_(False)
original_config=teacher.config.to_dict()

def make_record(text):
    ids=tokenizer(text,add_special_tokens=False)["input_ids"][:MAX_TOKENS+1]
    if len(ids)<8:raise ValueError("Text too short")
    return {"input_ids":torch.tensor([ids[:-1]],dtype=torch.long),
            "gold_next":int(ids[-1]),"text":text,"length":len(ids)-1}
train_rows=[make_record(s) for s in train_text]
dev_rows=[make_record(s) for s in dev_text]
# Holdout records are NOT yet materialized or evaluated.
@torch.inference_mode()
def attach_reference(records):
    for index,rec in enumerate(records):
        x=rec["input_ids"].to(device)
        logits=teacher(input_ids=x,use_cache=False,logits_to_keep=1).logits[:, -1]
        rec["teacher_logits"]=logits.float().cpu().contiguous()
        if index%25==0:print("Reference logits",index,"/",len(records))
    return records
train_rows=attach_reference(train_rows)
dev_rows=attach_reference(dev_rows)
teacher.cpu()
gc.collect();torch.cuda.empty_cache()
print("Teacher logits cached, teacher on CPU.")


In [ ]:
#@title 4. Replace ALL 24 token mixers; freeze original FFNs and layer norms
def ff_norm_fingerprint(model):
    sha=hashlib.sha256()
    for i,layer in enumerate(model.model.layers):
        for name in ("mlp","input_layernorm","post_attention_layernorm"):
            for k,x in getattr(layer,name).state_dict().items():
                sha.update(f"{i}/{name}/{k}".encode())
                sha.update(x.detach().cpu().contiguous().numpy().tobytes())
    return sha.hexdigest()

TEACHER_FFN_SHA=ff_norm_fingerprint(teacher)
student=copy.deepcopy(teacher)
original_student_layers=tuple(layer.block_type for layer in student.model.layers)
counts=wa.install_all24(student,window=WINDOW,anchors=ANCHORS,chunk_size=CHUNK)
student.to(device).eval().requires_grad_(False)
assert counts=={"delta_replaced":18,"full_replaced":6}
assert ff_norm_fingerprint(student)==TEACHER_FFN_SHA
assert not any("cenn" in name.lower() for name,_ in student.named_modules())
assert all(not p.requires_grad for p in student.parameters())
print("ALL 24 REPLACED:",counts,"original FFNs + norms exact SHA:",TEACHER_FFN_SHA)

@torch.inference_mode()
def metrics(model,rows):
    model.eval()
    n=0; agree=0; js=0.; kl=0.; student_ce=0.;teacher_ce=0.
    for rec in rows:
        x=rec["input_ids"].to(device)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(input_ids=x,use_cache=False,logits_to_keep=1).logits[:, -1]
        a=rec["teacher_logits"].to(device).float()
        b=logits.float()
        pa=a.softmax(-1);pb=b.softmax(-1)
        midpoint=0.5*(pa+pb)
        js+=float(0.5*((pa*(pa.clamp_min(1e-9).log()-midpoint.clamp_min(1e-9).log())).sum()
                        +(pb*(pb.clamp_min(1e-9).log()-midpoint.clamp_min(1e-9).log())).sum()))
        kl+=float(F.kl_div(b.log_softmax(-1),pa,reduction="sum"))
        agree+=int(a.argmax(-1).item()==b.argmax(-1).item())
        goal=torch.tensor([rec["gold_next"]],dtype=torch.long,device=device)
        student_ce+=float(F.cross_entropy(b,goal))
        teacher_ce+=float(F.cross_entropy(a,goal))
        n+=1
    return {"n":n,"top1_agreement":agree/n,"mean_js":js/n,
            "mean_kl":kl/n,"student_nll":student_ce/n,
            "teacher_nll":teacher_ce/n,
            "ppl_ratio":math.exp(min(30,(student_ce-teacher_ce)/n))}
BASELINE_DEV=metrics(student,dev_rows)
print("Untuned all-24 substitute DEV:",BASELINE_DEV)
(REPORTS/"architecture.json").write_text(json.dumps({
    "source":MODEL_ID,"source_layout":original_student_layers,
    "replaced":counts,"no_cenn":True,"no_delta_recurrence":True,
    "window":WINDOW,"anchors":ANCHORS,"chunk_size":CHUNK,
    "ffn_norm_sha256":TEACHER_FFN_SHA,"train_data":DATA_SOURCE
},indent=2))


In [ ]:
#@title 5. LoRA distillation of all 24 replacement mixers (FP16 + frozen FFNs)
class LoRA(nn.Module):
    def __init__(self,base,rank=LORA_RANK):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base.requires_grad_(False)
        d=base.weight.device
        self.A=nn.Parameter(torch.empty(rank,base.in_features,dtype=torch.float32,device=d))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,dtype=torch.float32,device=d))
        nn.init.normal_(self.A,std=0.01)
        self.scale=1/rank
    def forward(self,x):
        return self.base(x)+F.linear(F.linear(x,self.A),self.B)*self.scale
    @torch.no_grad()
    def merge(self):
        delta=(self.B.float()@self.A.float())*self.scale
        self.base.weight.add_(delta.to(self.base.weight.dtype))
        return self.base

trainable=[]
for layer in student.model.layers:
    module=(layer.linear_attn if layer.block_type=="linear_attention" else layer.self_attn)
    names=("in_proj_qkv","out_proj") if isinstance(module,wa.WindowAnchorDelta) else ("q_proj","k_proj","v_proj","o_proj")
    for name in names:
        wrapped=LoRA(getattr(module,name))
        setattr(module,name,wrapped)
        trainable.extend((wrapped.A,wrapped.B))
assert all(p.dtype==torch.float32 for p in trainable)
assert all(not p.requires_grad for layer in student.model.layers for p in layer.mlp.parameters())
assert all(not p.requires_grad for layer in student.model.layers
           for p in layer.input_layernorm.parameters())

optimizer=torch.optim.AdamW(trainable,lr=LR_START,weight_decay=0.0)
scaler=torch.amp.GradScaler("cuda")
best=[p.detach().cpu().clone() for p in trainable]
best_step=0
best_eval=BASELINE_DEV
history=[{"step":0,"dev":BASELINE_DEV}]
order=list(range(len(train_rows)))
rng=random.Random(SEED+50)
ix=0
student.eval()
def select_key(x):
    return (x["top1_agreement"],-x["mean_js"],-x["ppl_ratio"])

for step in range(1,TRAIN_STEPS+1):
    eta=(step-1)/max(1,TRAIN_STEPS-1)
    lr=LR_END+0.5*(LR_START-LR_END)*(1+math.cos(math.pi*eta))
    for g in optimizer.param_groups:g["lr"]=lr
    optimizer.zero_grad(set_to_none=True)
    parts=[]
    for _ in range(GRAD_ACCUM):
        if ix==0:rng.shuffle(order)
        rec=train_rows[order[ix]];ix=(ix+1)%len(order)
        tokens=rec["input_ids"].to(device)
        gold=torch.tensor([rec["gold_next"]],device=device)
        teacher_logits=rec["teacher_logits"].to(device).float()
        with torch.autocast("cuda",dtype=torch.float16):
            out=student(input_ids=tokens,use_cache=False,logits_to_keep=1)
            p=out.logits[:, -1].float()
            q=teacher_logits
            tau=DISTILL_TEMP
            kd=F.kl_div(F.log_softmax(p/tau,dim=-1),
                        F.softmax(q/tau,dim=-1),reduction="batchmean")*tau*tau
            ce=F.cross_entropy(p,gold)
            loss=kd+GOLD_CE_WEIGHT*ce
        if not bool(loss.isfinite().item()):
            raise FloatingPointError(f"Invalid loss at step {step}")
        scaler.scale(loss/GRAD_ACCUM).backward()
        parts.append(float(loss.detach()))
    scaler.unscale_(optimizer)
    norm=torch.nn.utils.clip_grad_norm_(trainable,MAX_GRAD_NORM)
    if bool(torch.isfinite(norm).item()):
        scaler.step(optimizer)
    else:print("Nonfinite gradient skipped:",step)
    scaler.update()
    if step%EVAL_EVERY==0 or step==1 or step==TRAIN_STEPS:
        current=metrics(student,dev_rows)
        if select_key(current)>select_key(best_eval):
            best_eval=current
            best_step=step
            best=[p.detach().cpu().clone() for p in trainable]
        record={"step":step,"lr":lr,"loss":float(np.mean(parts)),
                "dev":current,"best_step":best_step}
        history.append(record)
        (REPORTS/"training_history.json").write_text(json.dumps(history,indent=2))
        print(f"Step {step}/{TRAIN_STEPS}: teacher match {current['top1_agreement']:.3f}, "
              f"PPL ratio {current['ppl_ratio']:.3f}, best {best_step}")
with torch.no_grad():
    for param,snapshot in zip(trainable,best):
        param.copy_(snapshot.to(param.device))
for layer in student.model.layers:
    mod=layer.linear_attn if layer.block_type=="linear_attention" else layer.self_attn
    names=("in_proj_qkv","out_proj") if isinstance(mod,wa.WindowAnchorDelta) else ("q_proj","k_proj","v_proj","o_proj")
    for name in names:
        wrapper=getattr(mod,name)
        assert isinstance(wrapper,LoRA)
        setattr(mod,name,wrapper.merge())
del optimizer,best
gc.collect();torch.cuda.empty_cache()
assert ff_norm_fingerprint(student)==TEACHER_FFN_SHA
FINAL_DEV=metrics(student,dev_rows)
print("Best training step",best_step,"FINAL DEV",FINAL_DEV)


In [ ]:
#@title 6. Independent sealed HOLDOUT and honest FP16 latency measurements
# FIRST AND ONLY holdout access; never use it to choose training steps.
holdout_rows=[make_record(s) for s in holdout_text]
teacher.to(device).eval().requires_grad_(False)
holdout_rows=attach_reference(holdout_rows)
HOLDOUT=metrics(student,holdout_rows)
print("SEALED HOLDOUT",HOLDOUT)

@torch.inference_mode()
def median_prefill_ms(model,tokens,repeats=8):
    model.eval()
    x=tokens.to(device)
    for _ in range(2):
        with torch.autocast("cuda",dtype=torch.float16):
            model(input_ids=x,use_cache=False,logits_to_keep=1)
    torch.cuda.synchronize()
    durations=[]
    for _ in range(repeats):
        begin=torch.cuda.Event(enable_timing=True)
        end=torch.cuda.Event(enable_timing=True)
        begin.record()
        with torch.autocast("cuda",dtype=torch.float16):
            model(input_ids=x,use_cache=False,logits_to_keep=1)
        end.record();end.synchronize()
        durations.append(float(begin.elapsed_time(end)))
    return float(np.median(durations))

timings=[]
for length in (64,128,160):
    # One authentic holdout excerpt; pad by repetition (not zeros) if needed.
    tokens=holdout_rows[0]["input_ids"].to(device)
    if tokens.shape[1]<length:
        reps=math.ceil(length/tokens.shape[1])
        tokens=tokens.repeat(1,reps)[:,:length]
    else:tokens=tokens[:,:length]
    teacher_ms=median_prefill_ms(teacher,tokens)
    student_ms=median_prefill_ms(student,tokens)
    timings.append({"input_tokens":length,"teacher_ms":teacher_ms,
                    "student_ms":student_ms,
                    "speedup":teacher_ms/max(1e-9,student_ms),
                    "dtype":"float16","batch_size":1,"use_cache":False})
    print("Latency",timings[-1])
teacher.cpu();gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 7. Chat-quality tests (greedy, NO CACHE; text only)
TEST_MESSAGES=[
    "Explain in two sentences why the sky appears blue.",
    "What is 17 multiplied by 23? Answer with the number.",
    "Write one friendly greeting in German.",
    "Explain the difference between recurrent and full attention.",
    "Give two safety considerations when operating a small drone."
]
@torch.inference_mode()
def greedy_text(model,prompt,new_tokens=32):
    messages=[{"role":"user","content":prompt}]
    try:
        ins=tokenizer.apply_chat_template(
            messages,tokenize=True,add_generation_prompt=True,
            return_tensors="pt",enable_thinking=False)
    except TypeError:
        ins=tokenizer.apply_chat_template(
            messages,tokenize=True,add_generation_prompt=True,
            return_tensors="pt")
    if isinstance(ins,dict):ins=ins["input_ids"]
    ins=ins.to(device)
    initial=ins.shape[-1]
    for _ in range(new_tokens):
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(input_ids=ins,use_cache=False,logits_to_keep=1).logits[:,-1]
        next_id=logits.argmax(dim=-1,keepdim=True)
        ins=torch.cat((ins,next_id),dim=-1)
        if int(next_id[0,0]) in (tokenizer.eos_token_id,):
            break
    return tokenizer.decode(ins[0,initial:],skip_special_tokens=True)
chat_rows=[]
for prompt in TEST_MESSAGES:
    row={"prompt":prompt}
    try:
        row["student"]=greedy_text(student,prompt)
    except Exception as exc:
        row["error"]=repr(exc)
        print("Chat test failed (metrics/export will continue):",repr(exc))
    chat_rows.append(row)
    print("\nUSER:",prompt,"\nSUBSTITUTE:",row.get("student","FAILED"))
(REPORTS/"chat_examples.json").write_text(json.dumps(chat_rows,indent=2))


In [ ]:
#@title 8. Quality gate, FP16 save/reload parity, and local model files
median_speed=float(np.median([r["speedup"] for r in timings]))
qualified=bool(
    HOLDOUT["top1_agreement"]>=MIN_TEACHER_AGREEMENT
    and HOLDOUT["ppl_ratio"]<=MAX_PERPLEXITY_RATIO
    and HOLDOUT["mean_js"]<=MAX_JS
    and median_speed>=MIN_SPEEDUP)
checks={
    "minimum_top1_agreement":MIN_TEACHER_AGREEMENT,
    "maximum_perplexity_ratio":MAX_PERPLEXITY_RATIO,
    "maximum_js_divergence":MAX_JS,
    "minimum_median_speedup":MIN_SPEEDUP}
manifest={
    "architecture":"Qwen3.5_0.8B_All24_WindowAnchor_NoCeNN",
    "source_model":MODEL_ID,"dtype":"float16",
    "original_layers":24,"mixers_replaced":{"delta":18,"full":6},
    "attention_settings":{"window":WINDOW,"anchors":ANCHORS,"chunk_size":CHUNK},
    "no_cenn":True,"no_delta_recurrence":True,"no_kv_cache":True,
    "text_only":True,"ffn_norm_sha256":TEACHER_FFN_SHA,
    "training_steps":TRAIN_STEPS,"selected_step":best_step,
    "dataset":DATA_SOURCE,"dev":FINAL_DEV,
    "holdout":HOLDOUT,"benchmark":timings,
    "median_speedup":median_speed,
    "publication_gate":checks,"qualified":qualified,
    "utc":datetime.now(timezone.utc).isoformat()
}
(REPORTS/"final_report.json").write_text(json.dumps(manifest,indent=2))
print("QUALITY GATE:",qualified, "median speedup",median_speed)
print("Full model local export (even if quality gate fails)...")
reload_ok=False
parity=None
reload_error=None
try:
    wa.save_custom(student,tokenizer,ARTIFACTS,manifest)
    assert (ARTIFACTS/"model.safetensors").exists()
    before=student.to(device).eval()
    probe=dev_rows[0]["input_ids"].to(device)
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
        original_logits=before(input_ids=probe,use_cache=False,logits_to_keep=1).logits
    student.cpu()
    gc.collect();torch.cuda.empty_cache()
    reloaded,loaded_tokenizer,loaded_counts=wa.load_custom(ARTIFACTS,device=device)
    assert loaded_counts=={"delta_replaced":18,"full_replaced":6}
    assert ff_norm_fingerprint(reloaded)==TEACHER_FFN_SHA
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
        roundtrip_logits=reloaded(input_ids=probe,use_cache=False,logits_to_keep=1).logits
    parity=float((original_logits.float()-roundtrip_logits.float()).abs().max())
    assert parity<0.01, ("Model reload logit mismatch",parity)
    reload_ok=True
    del reloaded
    gc.collect();torch.cuda.empty_cache()
    print("STRICT 24-LAYER FP16 RELOAD PASSED, max difference:",parity)
except Exception as exc:
    # Never publish a model if its exported weights cannot be reloaded!
    qualified=False
    reload_error=repr(exc)
    print("EXPORT/RELOAD FAILED. HF upload disabled, but statistics will download:",repr(exc))
finally:
    manifest["roundtrip_max_logit_abs"]=parity
    manifest["reload_verified"]=reload_ok
    manifest["reload_error"]=reload_error
    manifest["qualified"]=bool(qualified and reload_ok)
    qualified=manifest["qualified"]
    (REPORTS/"final_report.json").write_text(json.dumps(manifest,indent=2))
    if (ARTIFACTS/"window_anchor_manifest.json").exists():
        (ARTIFACTS/"window_anchor_manifest.json").write_text(json.dumps(manifest,indent=2))


In [ ]:
#@title 9. Full model documentation, reproducibility and final statistics
card=f"""---
license: apache-2.0
library_name: pytorch
tags:
- qwen3.5
- window-anchor
- attention-substitution
- no-cenn
- research
---

# Qwen3.5-0.8B Window+Anchor ALL-24 (NO CeNN)

Text-only research checkpoint derived from {MODEL_ID} (Apache-2.0).

**18 DeltaNet and 6 full attention mixers replaced; FFNs/norms remain
unchanged.** Gated DeltaNet RECURRENCE is removed. Window/anchor attention
is **causal**. This implementation DOES NOT support KV-cache decoding,
images or video, and is NOT plug-and-play via AutoModel.from_pretrained.

Training examples: {TRAIN_EXAMPLES} | DEV: {DEV_EXAMPLES} | sealed HOLDOUT: {HOLDOUT_EXAMPLES}.
Best step: {best_step}.
Holdout top-1 teacher agreement: {HOLDOUT['top1_agreement']:.3%}.
Holdout NLL-based perplexity ratio: {HOLDOUT['ppl_ratio']:.3f}.
Median end-to-end FP16 prefill speedup: {median_speed:.3f}x.
Publication gate passed: {qualified}. Export reload verified: {reload_ok}.

## Load locally or from Hub

Install recent transformers, torch, and safetensors, then import
\`tinycenn_lm.qwen35_window_anchor.load_custom\` from the
[TinyCeNN-LM source](https://github.com/vtavakkoli/TinyCeNN-LM).

\`\`\`python
from tinycenn_lm.qwen35_window_anchor import load_custom
model, tokenizer, counts = load_custom("PATH_TO_DOWNLOADED_REPOSITORY",device="cuda")
inputs = tokenizer("Hello",return_tensors="pt").to("cuda")
out = model(**inputs,use_cache=False,logits_to_keep=1)
print(tokenizer.decode(out.logits[:,-1].argmax(-1)))
\`\`\`

For Hub download use huggingface_hub.snapshot_download and pass the
resulting local directory to load_custom. For chat, use full-sequence
greedy generation with \`use_cache=False\`. This model does NOT implement
native recurrent DeltaNet caches or incremental KV generation.
"""
(ARTIFACTS/"README.md").write_text(card)
(ARTIFACTS/"requirements.txt").write_text(
    "torch>=2.4\nsafetensors>=0.5\nhuggingface_hub>=0.34\n"
    "git+https://github.com/huggingface/transformers.git@main\n"
    "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@143f99a4bdb527001d0f5f8433f435535656a409\n")
(REPORTS/"summary.json").write_text(json.dumps({
    "qualified":qualified,"best_step":best_step,"final_dev":FINAL_DEV,
    "sealed_holdout":HOLDOUT,"timings":timings,"checks":checks,
    "fp16_reload_max_abs_error":parity},indent=2))


In [ ]:
#@title 10. ZIP all statistics and download automatically (BEFORE optional HF upload)
from IPython.display import display, FileLink
STATS_ZIP=BASE/"Qwen35_0p8B_All24_NoCeNN_Final_Statistics.zip"
MODEL_ZIP=BASE/"Qwen35_0p8B_All24_NoCeNN_Standalone_Model.zip"
report_paths=sorted(p for p in REPORTS.rglob("*") if p.is_file())
assert (REPORTS/"final_report.json") in report_paths
with zipfile.ZipFile(STATS_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in report_paths:
        z.write(p,"statistics/"+p.relative_to(REPORTS).as_posix())
if (ARTIFACTS/"model.safetensors").is_file():
    with zipfile.ZipFile(MODEL_ZIP,"w",allowZip64=True) as z:
        for p in sorted(p for p in ARTIFACTS.rglob("*") if p.is_file()):
            compression=(zipfile.ZIP_STORED if p.suffix==".safetensors"
                         else zipfile.ZIP_DEFLATED)
            z.write(p,"model/"+p.relative_to(ARTIFACTS).as_posix(),
                    compress_type=compression)
        for p in report_paths:
            z.write(p,"statistics/"+p.relative_to(REPORTS).as_posix(),
                    compress_type=zipfile.ZIP_DEFLATED)
else:
    print("No model.safetensors was exported; statistics ZIP will still download.")
for p in (STATS_ZIP,MODEL_ZIP):
    if not p.exists():continue
    with zipfile.ZipFile(p) as z:
        assert z.testzip() is None
    print("ZIP verified:",p,"MiB",round(p.stat().st_size/2**20,2))
display(FileLink(str(STATS_ZIP)))
if MODEL_ZIP.exists():display(FileLink(str(MODEL_ZIP)))
try:
    from google.colab import files
    if AUTO_DOWNLOAD_STATS:files.download(str(STATS_ZIP))
    if AUTO_DOWNLOAD_MODEL_ZIP and MODEL_ZIP.exists():files.download(str(MODEL_ZIP))
except Exception as exc:
    print("Browser auto-download failed:",repr(exc))
    print("Use the displayed FileLinks or Colab Files sidebar.")


In [ ]:
#@title 11. Hugging Face upload ONLY IF the model passed ALL quality checks
HF_STATUS={"qualified":qualified,"attempted":False,"uploaded":False,
           "reason":"Quality gate failed; local model and ZIPs were saved."}
if qualified and UPLOAD_IF_QUALIFIED:
    try:
        token=get_token()
        if not token:
            login()
            token=get_token()
        assert token,"Hugging Face write token needed"
        api=HfApi(token=token)
        username=api.whoami()["name"]
        repo_id=f"{username}/{HF_NAME}"
        HF_STATUS["attempted"]=True
        api.create_repo(repo_id,repo_type="model",private=HF_PRIVATE,exist_ok=True)
        api.upload_folder(
            repo_id=repo_id,repo_type="model",folder_path=str(ARTIFACTS),
            commit_message="Causal all-24 window-anchor substitution verified on sealed holdout")
        files_on_hf=set(api.list_repo_files(repo_id,repo_type="model"))
        required={"model.safetensors","window_anchor_manifest.json",
                  "qwen35_window_anchor.py","config.json","README.md"}
        assert required<=files_on_hf,(required-files_on_hf)
        HF_STATUS.update({"uploaded":True,"repo_id":repo_id,
                          "url":f"https://huggingface.co/{repo_id}","reason":"Quality gate passed"})
    except Exception as exc:
        HF_STATUS["reason"]=repr(exc)
        print("HF upload error; local model still saved:",repr(exc))
elif not UPLOAD_IF_QUALIFIED:
    HF_STATUS["reason"]="Upload disabled by user."
(REPORTS/"hf_upload_status.json").write_text(json.dumps(HF_STATUS,indent=2))
print("HUGGING FACE STATUS",HF_STATUS)
print("The statistics and model ZIP files are available regardless of upload status.")
